# Map Dataset to Reference

## Download and Process Dataset of Interest

Prior to predicting kinase activities, datasets need to be mapped to the KSTAR reference phosphoproteome to match the information contained within the KSTAR kinase-substrate networks. It is recommended to use the peptide sequence rather than the site number when possible, as this is more likely to be found in the most recent version of KSTAR reference. 

For this tutorial, we will be using a dataset published by (Chylek, 2014), but you can use any processed dataset. Make sure it includes UniProt accessions and appropriately formatted peptides/sites as described in the 'Preparing your Data' section. See below for an example of what the data should look like.


Reference:  L. A. Chylek, V. Akimov, J. Dengjel, K. T. G. Rigbolt, B. Hu, W. S. Hlavacek, and B. Blagoev.
Phosphorylation Site Dynamics of Early T-cell Receptor Signaling. PLoS ONE, 9(8):e104240, 2014.

In [1]:
#import KSTAR and other necesary packages
import pandas as pd

#load data
df = pd.read_csv('example.tsv', index_col = 0, sep = '\t')
df.head()

,query_accession,mod_sites,peptide,data:time:5,data:time:15,data:time:30,data:time:60,KSTAR_ACCESSION,KSTAR_PEPTIDE,KSTAR_SITE
MS_id,,,,,,,,,,
7605136,Q9P2D3-1,Y1104,EAAEVCEyAMSLAK,-0.01,-0.28,-0.03,-0.27,Q9P2D3,EAAEVCEyAMSLAK,NaN
7605137,A0FGR8-6,Y845,NLIAFSEDGSDPyVR,0.26,0.27,0.04,0.05,A0FGR8,NLIAFSEDGSDPyVR,NaN
7605138,Q5T4S7-2,Y5156,HNDMPIyEAADK,0.31,-0.15,0.01,-0.23,Q5T4S7,HNDMPIyEAADK,NaN
7605139,Q16181-1,Y30,NLEGyVGFANLPNQVYR,-0.14,-0.19,0.07,0.15,Q16181,NLEGyVGFANLPNQVYR,NaN
7605140,Q16181-1,Y41,NLEGYVGFANLPNQVyR,-0.14,-0.09,0.04,-0.06,Q16181,NLEGYVGFANLPNQVyR,NaN


Notice that all data columns in the dataset have 'data:' in front of them. This is how KSTAR will identify which columns to use when making evidence decisions. This can be done manually prior to mapping, or will be done by KSTAR automatically once you indicate which columns you would like to use as evidence.

## Map the Dataset to Reference

Before running KSTAR, you need to map your dataset to the reference phosphoproteome used by KSTAR to ensure site positions agree with the kinase-substrate information. First, we need to indicate where to save the mapped data and the name you would like to use for output files:

In [2]:
#define the directory where mapped dataset and run information will be saved. 
odir = './example'
name = 'example_run'

Next, you need to indicate where to find columns containing peptide-specific information. Construct a dictionary which indicates the columns where KSTAR can find information about each phosphopeptide. This should include:
- *accession_id*: the UniProt accession corresponding to the identified peptide. 

and either:
- *peptide*: amino acid sequence with phosphorylation sites lowercased
- *site*: modified amino acid + modification location, such as Y11

It is recommended to use the peptide sequence when possible.

In [3]:
#setup mapping columns: since we only have peptide column in the example data, we will use that column (and not provide a site column)
accession_col = 'query_accession'
peptide_col = 'peptide'
mapDict = {'accession_id':'query_accession', 'peptide':'peptide'}

In [4]:
from kstar import mapping
#map dataset and record process in the logger
exp_mapper = mapping.ExperimentMapper(experiment = df,
                                      columns = mapDict, 
                                      odir = odir,
                                      name = name)

Processing provided accessions...
Aligning peptides/sites to reference sequences...


Mapping peptides/sites to reference sequences:   0%|                                                                                                                | 0/689 [00:00<?, ?it/s]

Mapping peptides/sites to reference sequences:  19%|███████████████████▍                                                                                | 134/689 [00:00<00:00, 1329.80it/s]

Mapping peptides/sites to reference sequences:  39%|██████████████████████████████████████▊                                                             | 267/689 [00:00<00:00, 1252.70it/s]

Mapping peptides/sites to reference sequences:  60%|███████████████████████████████████████████████████████████▌                                        | 410/689 [00:00<00:00, 1330.10it/s]

Mapping peptides/sites to reference sequences:  80%|███████████████████████████████████████████████████████████████████████████████▊                    | 550/689 [00:00<00:00, 1354.54it/s]

Mapping peptides/sites to reference sequences: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 689/689 [00:00<00:00, 1401.88it/s]

Mapping complete.


If you look at the ExperimentMapper class, you will find that five new columns have been added to the original dataset, which allows for easy mapping to KSTAR networks used in activity prediction.

In [5]:
exp_mapper.experiment.head()

,query_accession,mod_sites,peptide,data:time:5,data:time:15,data:time:30,data:time:60,KSTAR_ACCESSION,KSTAR_PEPTIDE,KSTAR_SITE,KSTAR_NUM_COMPENDIA,KSTAR_NUM_COMPENDIA_CLASS
0,Q9P2D3-1,Y1104,EAAEVCEyAMSLAK,-0.01,-0.28,-0.03,-0.27,Q9P2D3,EAAEVCEyAMSLAKN,Y1104,0,0
1,A0FGR8-6,Y845,NLIAFSEDGSDPyVR,0.26,0.27,0.04,0.05,A0FGR8,SEDGSDPyVRMYLLP,Y824,2,1
2,Q5T4S7-2,Y5156,HNDMPIyEAADK,0.31,-0.15,0.01,-0.23,Q5T4S7,RHNDMPIyEAADKAL,Y5135,1,1
3,Q16181-1,Y30,NLEGyVGFANLPNQVYR,-0.14,-0.19,0.07,0.15,Q16181,QQKNLEGyVGFANLP,Y30,5,2
4,Q16181-1,Y41,NLEGYVGFANLPNQVyR,-0.14,-0.09,0.04,-0.06,Q16181,ANLPNQVyRKSVKRG,Y41,1,1


These additional columns have the following meaning:

1. *KSTAR_ACCESSION*: Uniprot accession id corresponding to reviewed protein sequence, focusing only on the canonical isoforms of each protein.
2. *KSTAR_PEPTIDE*: Peptide sequence containing a single phosphorylation site, including the 7 amino acids both before and after the modified residue.
3. *KSTAR_SITE*: Location of modified residue in the protein
4. *KSTAR_NUM_COMPENDIA*: The number of different phosphoproteome compendia that modification site is identified in, used as an indicator of the study bias of each modification site. For this purpose, PhosphoSitePlus, PhosphoELM, HPRD, ______, and ProteomeScout were profiled.
5. *KSTAR_NUM_COMPENDIA_CLASS*: Same as 4, but sites are grouped into smaller classes based on study bias (0 is <1 compendia, 1 is 1-3 compendia, 2 is >3 compendia)

## Save Mapping Results

You can save the result using the `save_experiment()` function. This will save the following files in a MAPPED_DATA directory:
- *name_mapped.csv*: dataset mapped to reference phosphoproteome, to be used for activity prediction
- *name_mapping_stats.txt*: summary of how successful mapping process was
- *name_missed_sites.csv*: dataframe of all sites that were removed from the dataset and the reason

Looking at the mapping_stats.txt file is a good check to make sure the data was effectively mapped:

In [6]:
exp_mapper.save_experiment()

You can either inspect the mapping_stats.txt file directly or load it into python to look at results

In [7]:
#load mapped data
stats_file = f"{odir}/MAPPED_DATA/{name}_mapping_stats.txt"
with open(stats_file, 'r') as f:
    print(f.read())

Site counts per data column after mapping:

Phospho type: Y
data:time:5 -> 663
data:time:15 -> 663
data:time:30 -> 663
data:time:60 -> 663

Phospho type: ST
data:time:5 -> 1
data:time:15 -> 1
data:time:30 -> 1
data:time:60 -> 1

Mapping success statistics:
Mapped Peptides: 653/653 peptides mapped (100.00%).

Reasons for unmapped sites/peptides:

See ./example/MAPPED_DATA/example_run_removed_sites.csv for details on removed sites/peptides.



You are now ready to proceed to activity calculation